# S4 — Guided V4 feature selection, modelling, calibration, and robustness

Run each section in order. The notebook proposes defaults, but every decision cell can be edited before the following stage is run.


In [ ]:
from pathlib import Path
import os, sys, joblib, pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
EVENTS_DIR, F360_DIR, OUTPUT_DIR = Path(os.environ['EVENTS_DIR']), Path(os.environ['F360_DIR']), Path(os.environ['OUTPUT_DIR'])
table = pd.read_parquet(OUTPUT_DIR / 'v4_features.parquet')


## 1. Data and feature-quality checks


In [ ]:
from src.hpn_features import V4_ALL_FEATURES, V4_TOP_16
from src.hpn_model import feature_diagnostics

print('Rows:', len(table), 'matches:', table['match_id'].nunique(), 'sequences:', table[['match_id','seq_id']].drop_duplicates().shape[0])
display(table['outcome_tag'].value_counts(dropna=False).to_frame('anchors'))
diagnostics = feature_diagnostics(table, V4_ALL_FEATURES)
display(diagnostics['missing'])
display(diagnostics['correlation_pairs'].head(20))


## 2. User feature decision

Edit exclusions or replace the proposed final V4-top-16 list, then run the next cell.


In [ ]:
USER_EXCLUDED_FEATURES = []
FINAL_FEATURES = [feature for feature in V4_TOP_16 if feature not in USER_EXCLUDED_FEATURES]
print('Selected features:', FINAL_FEATURES)


## 3. Grouped model comparison

The split unit is match_id, preventing frames from the same match appearing in both training and validation folds.


In [ ]:
from src.hpn_model import grouped_model_comparison

MODEL_FOLDS = 5
comparison, fitted = grouped_model_comparison(table, FINAL_FEATURES, folds=MODEL_FOLDS)
display(comparison)
RECOMMENDED_MODEL = comparison.iloc[0]['model']
print('Recommended by grouped log-loss:', RECOMMENDED_MODEL)


## 4. User model and calibration decision


In [ ]:
SELECTED_MODEL = RECOMMENDED_MODEL  # logistic, xgb_unweighted, or xgb_balanced
CALIBRATION = 'raw'                   # raw or isotonic
print(SELECTED_MODEL, CALIBRATION)


In [ ]:
from src.hpn_model import CLASS_NAMES, fit_isotonic, save_bundle

selected = fitted[SELECTED_MODEL]
calibrators = None
if CALIBRATION == 'isotonic':
    y = table['outcome_tag'].map({name: i for i, name in enumerate(CLASS_NAMES)}).to_numpy()
    calibrators = fit_isotonic(selected['oof_probability'], y)
bundle_path = OUTPUT_DIR / 'v4_model_bundle.joblib'
save_bundle(bundle_path, selected['model'], FINAL_FEATURES, CALIBRATION, calibrators)
print('Saved:', bundle_path)


## 5. Robustness testing

Set RUN_ROBUSTNESS to True after the model definition is fixed. Each scenario rebuilds the user-data V4 table under changed network parameters and reports feature stability.


In [ ]:
RUN_ROBUSTNESS = False
ROBUSTNESS_SCENARIOS = {
    'player_reach_low': {'player_distance': 5.76, 'player_sd': 1.98},
    'player_reach_high': {'player_distance': 7.04, 'player_sd': 2.42},
    'boundary_low': {'boundary_distance': 2.56, 'boundary_sd': .88},
    'boundary_high': {'boundary_distance': 3.84, 'boundary_sd': 1.32},
}
if RUN_ROBUSTNESS:
    from src.hpn_network import PressureParams
    from src.hpn_features import build_v4_feature_table
    from src.hpn_model import run_robustness
    labels = pd.read_csv(OUTPUT_DIR / 'labels.csv')
    def rebuild(overrides):
        base = PressureParams()
        return build_v4_feature_table(labels, EVENTS_DIR, F360_DIR, PressureParams(**{**base.__dict__, **overrides}))
    robustness = run_robustness(table, rebuild, ROBUSTNESS_SCENARIOS, FINAL_FEATURES)
    display(robustness.groupby('scenario')[['spearman','mean_abs_change']].mean())
